In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

ruta = Path(r"C:\Users\yeral\Documents\M5 Walmart")

print("Proyecto:", ruta)

Proyecto: C:\Users\yeral\Documents\M5 Walmart


In [2]:
ruta_models = ruta / "models"
ruta_outputs = ruta / "outputs"

ruta_models.mkdir(parents=True, exist_ok=True)
ruta_outputs.mkdir(parents=True, exist_ok=True)

print("Models:", ruta_models)
print("Outputs:", ruta_outputs)

Models: C:\Users\yeral\Documents\M5 Walmart\models
Outputs: C:\Users\yeral\Documents\M5 Walmart\outputs


In [4]:
sales = pd.read_csv(
    ruta / "data" / "raw" / "sales_train_validation.csv"
)

segmentacion = pd.read_csv(
    ruta / "data" / "processed" / "segmentacion_sku_tienda.csv"
)

print("Sales:", sales.shape)
print("Segmentación:", segmentacion.shape)

Sales: (30490, 1919)
Segmentación: (30490, 7)


In [6]:
train_ml_muestra = pd.read_csv(
    ruta / "data" / "processed" / "train_ml.csv",
    nrows=5
)

print(train_ml_muestra.columns.tolist())
print(train_ml_muestra.shape)

['id', 'item_id', 'store_id', 'macrogrupo', 'd', 'demanda', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_35', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_56', 'date', 'day_of_week', 'month', 'event']
(5, 19)


In [8]:
from xgboost import XGBRegressor

chunks = []

for chunk in pd.read_csv(
    ruta / "data" / "processed" / "train_ml.csv",
    chunksize=500_000
):
    chunk = chunk[chunk["macrogrupo"] == "Estacional"]
    
    if not chunk.empty:
        chunks.append(chunk)

train_estacional = pd.concat(chunks, ignore_index=True)

X_estacional = train_estacional[feature_cols]
y_estacional = train_estacional["demanda"]

print(X_estacional.shape)
print(y_estacional.shape)

(766941, 12)
(766941,)


In [12]:
mapa_dias = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

X_estacional = X_estacional.copy()

X_estacional["day_of_week"] = X_estacional["day_of_week"].map(mapa_dias)

print(X_estacional.dtypes)
print("Valores nulos:", X_estacional.isna().sum().sum())

lag_7              float64
lag_14             float64
lag_21             float64
lag_28             float64
lag_35             float64
lag_56             float64
rolling_mean_7     float64
rolling_mean_28    float64
rolling_mean_56    float64
day_of_week          int64
month                int64
event                int64
dtype: object
Valores nulos: 0


In [13]:
modelo_xgb_estacional = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

modelo_xgb_estacional.fit(X_estacional, y_estacional)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [14]:
joblib.dump(
    modelo_xgb_estacional,
    ruta_models / "xgb_estacional.joblib"
)

print("Modelo Estacional guardado correctamente.")

Modelo Estacional guardado correctamente.


In [15]:
calendar = pd.read_csv(
    ruta / "data" / "raw" / "calendar.csv"
)

calendar["date"] = pd.to_datetime(calendar["date"])

sales_eval = pd.read_csv(
    ruta / "data" / "raw" / "sales_train_evaluation.csv"
)

print("Calendar:", calendar.shape)
print("Sales evaluation:", sales_eval.shape)

Calendar: (1969, 14)
Sales evaluation: (30490, 1947)


In [16]:
# Series estacionales
ids_estacionales = segmentacion.loc[
    segmentacion["macrogrupo"] == "Estacional", "id"
].tolist()

print("Series estacionales:", len(ids_estacionales))

# Historial: d_1 a d_1913
ventas_hist = sales.loc[
    sales["id"].isin(ids_estacionales)
].copy()

# Evaluación real: d_1914 a d_1941
dias_futuros = [f"d_{i}" for i in range(1914, 1942)]

ventas_reales = sales_eval.loc[
    sales_eval["id"].str.replace("_evaluation", "_validation").isin(ids_estacionales),
    dias_futuros
]

print("Historial:", ventas_hist.shape)
print("Reales:", ventas_reales.shape)

Series estacionales: 413
Historial: (413, 1919)
Reales: (413, 28)


In [17]:
def pronostico_recursivo_batch(modelo, historial, calendario_futuro):
    historial = historial.astype(np.float32).copy()
    predicciones = []

    for _, dia in calendario_futuro.iterrows():
        X_futuro = pd.DataFrame({
            "lag_7": historial[:, -7],
            "lag_14": historial[:, -14],
            "lag_21": historial[:, -21],
            "lag_28": historial[:, -28],
            "lag_35": historial[:, -35],
            "lag_56": historial[:, -56],
            "rolling_mean_7": historial[:, -7:].mean(axis=1),
            "rolling_mean_28": historial[:, -28:].mean(axis=1),
            "rolling_mean_56": historial[:, -56:].mean(axis=1),
            "day_of_week": [dia["day_of_week"]] * len(historial),
            "month": [dia["month"]] * len(historial),
            "event": [dia["event"]] * len(historial)
        })

        pred = modelo.predict(X_futuro)
        pred = np.maximum(pred, 0)

        predicciones.append(pred)
        historial = np.column_stack((historial, pred))

    return np.column_stack(predicciones)

In [18]:
calendario_futuro = calendar[
    calendar["d"].isin(dias_futuros)
].copy()

calendario_futuro["day_of_week"] = calendario_futuro["weekday"].map({
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
})

calendario_futuro["event"] = (
    calendario_futuro["event_name_1"].notna() |
    calendario_futuro["event_name_2"].notna()
).astype(int)

calendario_futuro = calendario_futuro.sort_values("d")

print(calendario_futuro[[
    "d", "date", "weekday", "day_of_week", "month", "event"
]])

           d       date    weekday  day_of_week  month  event
1913  d_1914 2016-04-25     Monday            0      4      0
1914  d_1915 2016-04-26    Tuesday            1      4      0
1915  d_1916 2016-04-27  Wednesday            2      4      0
1916  d_1917 2016-04-28   Thursday            3      4      0
1917  d_1918 2016-04-29     Friday            4      4      0
1918  d_1919 2016-04-30   Saturday            5      4      1
1919  d_1920 2016-05-01     Sunday            6      5      1
1920  d_1921 2016-05-02     Monday            0      5      0
1921  d_1922 2016-05-03    Tuesday            1      5      0
1922  d_1923 2016-05-04  Wednesday            2      5      0
1923  d_1924 2016-05-05   Thursday            3      5      1
1924  d_1925 2016-05-06     Friday            4      5      0
1925  d_1926 2016-05-07   Saturday            5      5      0
1926  d_1927 2016-05-08     Sunday            6      5      1
1927  d_1928 2016-05-09     Monday            0      5      0
1928  d_

In [19]:
historial_estacional = ventas_hist[
    [f"d_{i}" for i in range(1, 1914)]
].to_numpy(dtype=np.float32)

print("Historial:", historial_estacional.shape)

Historial: (413, 1913)


In [20]:
predicciones_estacional = pronostico_recursivo_batch(
    modelo_xgb_estacional,
    historial_estacional,
    calendario_futuro
)

print("Pronóstico:", predicciones_estacional.shape)

Pronóstico: (413, 28)


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Valores reales
reales_estacional = ventas_reales.to_numpy(dtype=np.float32)

# Aplanar para comparar observación por observación
y_real = reales_estacional.ravel()
y_pred = predicciones_estacional.ravel()

mae = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))

# MAPE: solo donde la demanda real > 0
mask = y_real > 0
mape = np.mean(
    np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])
) * 100

print(f"Series: {predicciones_estacional.shape[0]:,}")
print(f"Observaciones: {predicciones_estacional.size:,}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

Series: 413
Observaciones: 11,564
MAE:  2.3759
RMSE: 3.9708
MAPE: 50.99%


# Filtrar únicamente las observaciones del macrogrupo Tendencia.
# Se utiliza lectura por bloques para no cargar en memoria todo train_ml.csv.

In [22]:
chunks = []

for chunk in pd.read_csv(
    ruta / "data" / "processed" / "train_ml.csv",
    chunksize=500_000
):
    chunk = chunk[chunk["macrogrupo"] == "Tendencia"]
    
    if not chunk.empty:
        chunks.append(chunk)

train_tendencia = pd.concat(chunks, ignore_index=True)

X_tendencia = train_tendencia[feature_cols].copy()
y_tendencia = train_tendencia["demanda"]

# Convertir día de la semana a numérico
mapa_dias = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

X_tendencia["day_of_week"] = X_tendencia["day_of_week"].map(mapa_dias)

print("X Tendencia:", X_tendencia.shape)
print("y Tendencia:", y_tendencia.shape)
print("Valores nulos:", X_tendencia.isna().sum().sum())

X Tendencia: (4176393, 12)
y Tendencia: (4176393,)
Valores nulos: 0


### Entrenamiento del modelo XGBoost — Tendencia

Se entrena un modelo XGBoost para las series clasificadas como Tendencia utilizando las 12 variables predictoras definidas previamente. Se mantienen los mismos hiperparámetros utilizados durante la experimentación para garantizar la comparabilidad de los resultados.

In [23]:
from xgboost import XGBRegressor

modelo_xgb_tendencia = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

modelo_xgb_tendencia.fit(
    X_tendencia,
    y_tendencia
)

print("Entrenamiento XGBoost Tendencia terminado.")

Entrenamiento XGBoost Tendencia terminado.


### Guardado del modelo XGBoost — Tendencia

Se guarda el modelo entrenado para conservar la versión final utilizada en la evaluación y posteriormente en el pronóstico de las series Tendencia.

In [24]:
joblib.dump(
    modelo_xgb_tendencia,
    ruta_models / "xgb_tendencia.joblib"
)

print("Modelo XGBoost Tendencia guardado correctamente.")

Modelo XGBoost Tendencia guardado correctamente.


### Pronóstico recursivo — Tendencia

Se genera un pronóstico de 28 días para las series clasificadas como Tendencia. El modelo utiliza las predicciones generadas en días anteriores para construir las variables rezagadas y medias móviles de los días siguientes, reproduciendo el esquema de pronóstico recursivo utilizado durante la experimentación.

In [25]:
# Identificar las series Tendencia y preparar su historial de 1,913 días.

ids_tendencia = segmentacion.loc[
    segmentacion["macrogrupo"] == "Tendencia", "id"
].tolist()

ventas_hist_tendencia = sales.loc[
    sales["id"].isin(ids_tendencia)
].copy()

print("Series Tendencia:", len(ids_tendencia))
print("Historial:", ventas_hist_tendencia.shape)

Series Tendencia: 2249
Historial: (2249, 1919)


In [26]:
# Convertir el historial de ventas de las series Tendencia a una matriz numérica
# para utilizarla como punto de partida del pronóstico recursivo.

historial_tendencia = ventas_hist_tendencia[
    [f"d_{i}" for i in range(1, 1914)]
].to_numpy(dtype=np.float32)

print("Historial Tendencia:", historial_tendencia.shape)

Historial Tendencia: (2249, 1913)


In [27]:
# Generar el pronóstico recursivo de 28 días para las series Tendencia.

predicciones_tendencia = pronostico_recursivo_batch(
    modelo_xgb_tendencia,
    historial_tendencia,
    calendario_futuro
)

print("Pronóstico Tendencia:", predicciones_tendencia.shape)

Pronóstico Tendencia: (2249, 28)


In [28]:
# Evaluar el pronóstico Tendencia contra la demanda real de los 28 días de evaluación.

reales_tendencia = sales_eval.loc[
    sales_eval["id"].str.replace("_evaluation", "_validation").isin(ids_tendencia),
    dias_futuros
].to_numpy(dtype=np.float32)

y_real = reales_tendencia.ravel()
y_pred = predicciones_tendencia.ravel()

mae_tendencia = mean_absolute_error(y_real, y_pred)
rmse_tendencia = np.sqrt(mean_squared_error(y_real, y_pred))

# Calcular MAPE únicamente para observaciones con demanda real mayor que cero.
mask = y_real > 0

mape_tendencia = np.mean(
    np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])
) * 100

print(f"Series: {predicciones_tendencia.shape[0]:,}")
print(f"Observaciones: {predicciones_tendencia.size:,}")
print(f"MAE:  {mae_tendencia:.4f}")
print(f"RMSE: {rmse_tendencia:.4f}")
print(f"MAPE: {mape_tendencia:.2f}%")

Series: 2,249
Observaciones: 62,972
MAE:  1.6680
RMSE: 2.9260
MAPE: 53.42%


In [29]:
# Preparar las series Sin patrón dominante y un lector por bloques para entrenar
# XGBoost sin cargar los 51.7 millones de registros completos en memoria.

ids_sin_patron = segmentacion.loc[
    segmentacion["macrogrupo"] == "Sin patrón dominante", "id"
].tolist()

print("Series Sin patrón dominante:", len(ids_sin_patron))

Series Sin patrón dominante: 27828


In [30]:
# Crear un iterador que lee train_ml.csv por bloques y conserva únicamente
# las observaciones del macrogrupo Sin patrón dominante para el entrenamiento.

import xgboost as xgb

class M5DataIter(xgb.DataIter):
    def __init__(self, ruta_archivo, macrogrupo, feature_cols, chunksize=500_000):
        self.ruta_archivo = ruta_archivo
        self.macrogrupo = macrogrupo
        self.feature_cols = feature_cols
        self.chunksize = chunksize
        self._it = None

        super().__init__()

    def reset(self):
        self._it = pd.read_csv(
            self.ruta_archivo,
            usecols=feature_cols + ["demanda", "macrogrupo"],
            chunksize=self.chunksize
        )

    def next(self, input_data):
        try:
            while True:
                chunk = next(self._it)

                chunk = chunk[
                    chunk["macrogrupo"] == self.macrogrupo
                ]

                if chunk.empty:
                    continue

                X = chunk[self.feature_cols].copy()

                # Convertir day_of_week a valores numéricos.
                X["day_of_week"] = X["day_of_week"].map({
                    "Monday": 0,
                    "Tuesday": 1,
                    "Wednesday": 2,
                    "Thursday": 3,
                    "Friday": 4,
                    "Saturday": 5,
                    "Sunday": 6
                })

                y = chunk["demanda"]

                input_data(
                    data=X,
                    label=y
                )

                return True

        except StopIteration:
            return False


iterador_sin_patron = M5DataIter(
    ruta / "data" / "processed" / "train_ml.csv",
    "Sin patrón dominante",
    feature_cols,
    chunksize=500_000
)

print("Iterador creado correctamente.")

Iterador creado correctamente.


In [31]:
# Crear la matriz de entrenamiento externa de XGBoost y verificar que contenga
# las 51,676,596 observaciones correspondientes a Sin patrón dominante.

dtrain_ext = xgb.ExtMemQuantileDMatrix(
    iterador_sin_patron,
    max_bin=256
)

print("Matriz externa creada correctamente.")
print("Registros esperados: 51,676,596")

Matriz externa creada correctamente.
Registros esperados: 51,676,596


In [32]:
# Entrenar XGBoost para las series Sin patrón dominante utilizando la matriz
# externa para manejar el gran volumen de observaciones sin cargar todo en RAM.

modelo_xgb_sin_patron = xgb.train(
    {
        "objective": "reg:squarederror",
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "seed": 42,
        "tree_method": "hist"
    },
    dtrain_ext,
    num_boost_round=100
)

print("Entrenamiento XGBoost Sin patrón dominante terminado.")

Entrenamiento XGBoost Sin patrón dominante terminado.


In [33]:
# Guardar el modelo XGBoost entrenado para las series Sin patrón dominante.

joblib.dump(
    modelo_xgb_sin_patron,
    ruta_models / "xgb_sin_patron.joblib"
)

print("Modelo XGBoost Sin patrón dominante guardado correctamente.")

Modelo XGBoost Sin patrón dominante guardado correctamente.


In [34]:
# Identificar y preparar el historial de 1,913 días de las series Sin patrón dominante.

ventas_hist_sin_patron = sales.loc[
    sales["id"].isin(ids_sin_patron),
    [f"d_{i}" for i in range(1, 1914)]
].copy()

historial_sin_patron = ventas_hist_sin_patron.to_numpy(
    dtype=np.float32
)

print("Series Sin patrón dominante:", historial_sin_patron.shape[0])
print("Historial:", historial_sin_patron.shape)

Series Sin patrón dominante: 27828
Historial: (27828, 1913)


In [36]:
# Definir el pronóstico recursivo para el modelo Booster de XGBoost,
# utilizando DMatrix para mantener compatibilidad con xgb.train().

def pronostico_recursivo_booster(modelo, historial, calendario_futuro):
    historial = historial.astype(np.float32).copy()
    predicciones = []

    for _, dia in calendario_futuro.iterrows():

        X_futuro = np.column_stack([
            historial[:, -7],
            historial[:, -14],
            historial[:, -21],
            historial[:, -28],
            historial[:, -35],
            historial[:, -56],
            historial[:, -7:].mean(axis=1),
            historial[:, -28:].mean(axis=1),
            historial[:, -56:].mean(axis=1),
            np.full(len(historial), dia["day_of_week"]),
            np.full(len(historial), dia["month"]),
            np.full(len(historial), dia["event"])
        ]).astype(np.float32)

        X_futuro = xgb.DMatrix(
            X_futuro,
            feature_names=feature_cols
        )

        pred = modelo.predict(X_futuro)
        pred = np.maximum(pred, 0)

        predicciones.append(pred)
        historial = np.column_stack((historial, pred))

    return np.column_stack(predicciones)

In [37]:
# Generar el pronóstico recursivo de 28 días para las series Sin patrón dominante.

predicciones_sin_patron = pronostico_recursivo_booster(
    modelo_xgb_sin_patron,
    historial_sin_patron,
    calendario_futuro
)

print("Pronóstico Sin patrón dominante:", predicciones_sin_patron.shape)

Pronóstico Sin patrón dominante: (27828, 28)


In [38]:
# Evaluar el pronóstico Sin patrón dominante contra la demanda real de los 28 días de evaluación.

reales_sin_patron = sales_eval.loc[
    sales_eval["id"].str.replace("_evaluation", "_validation").isin(ids_sin_patron),
    dias_futuros
].to_numpy(dtype=np.float32)

y_real = reales_sin_patron.ravel()
y_pred = predicciones_sin_patron.ravel()

mae_sin_patron = mean_absolute_error(y_real, y_pred)
rmse_sin_patron = np.sqrt(mean_squared_error(y_real, y_pred))

# Calcular MAPE únicamente para observaciones con demanda real mayor que cero.
mask = y_real > 0

mape_sin_patron = np.mean(
    np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])
) * 100

print(f"Series: {predicciones_sin_patron.shape[0]:,}")
print(f"Observaciones: {predicciones_sin_patron.size:,}")
print(f"MAE:  {mae_sin_patron:.4f}")
print(f"RMSE: {rmse_sin_patron:.4f}")
print(f"MAPE: {mape_sin_patron:.2f}%")

Series: 27,828
Observaciones: 779,184
MAE:  0.9648
RMSE: 2.0633
MAPE: 57.51%


In [43]:
# Convertir el pronóstico Estacional a formato largo y guardarlo como CSV
# para utilizarlo posteriormente en GitHub y en la aplicación Streamlit.

ids_estacionales_ordenados = ids_estacionales

filas_estacionales = sales.loc[
    sales["id"].isin(ids_estacionales_ordenados),
    ["id", "item_id", "store_id"]
].copy()

# Mantener el mismo orden utilizado para generar las predicciones.
filas_estacionales = (
    filas_estacionales
    .set_index("id")
    .loc[ids_estacionales_ordenados]
    .reset_index()
)

pronostico_estacional_largo = pd.DataFrame({
    "item_id": np.repeat(
        filas_estacionales["item_id"].to_numpy(),
        28
    ),
    "store_id": np.repeat(
        filas_estacionales["store_id"].to_numpy(),
        28
    ),
    "macrogrupo": "Estacional",
    "date": np.tile(
        calendario_futuro["date"].to_numpy(),
        len(filas_estacionales)
    ),
    "demanda_pronosticada": predicciones_estacional.ravel(),
    "modelo": "XGBoost"
})

ruta_forecast_estacional = ruta_outputs / "forecast_estacional.csv"

pronostico_estacional_largo.to_csv(
    ruta_forecast_estacional,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo:", ruta_forecast_estacional)
print("Filas:", len(pronostico_estacional_largo))
print(pronostico_estacional_largo.head())

Archivo: C:\Users\yeral\Documents\M5 Walmart\outputs\forecast_estacional.csv
Filas: 11564
         item_id store_id  macrogrupo       date  demanda_pronosticada  \
0  HOBBIES_1_004     CA_1  Estacional 2016-04-25              1.915244   
1  HOBBIES_1_004     CA_1  Estacional 2016-04-26              1.437366   
2  HOBBIES_1_004     CA_1  Estacional 2016-04-27              1.542311   
3  HOBBIES_1_004     CA_1  Estacional 2016-04-28              1.747647   
4  HOBBIES_1_004     CA_1  Estacional 2016-04-29              2.069149   

    modelo  
0  XGBoost  
1  XGBoost  
2  XGBoost  
3  XGBoost  
4  XGBoost  


In [44]:
# Convertir el pronóstico Tendencia a formato largo y guardarlo como CSV
# para utilizarlo posteriormente en GitHub y en la aplicación Streamlit.

filas_tendencia = sales.loc[
    sales["id"].isin(ids_tendencia),
    ["id", "item_id", "store_id"]
].copy()

# Mantener el mismo orden utilizado para generar las predicciones.
filas_tendencia = (
    filas_tendencia
    .set_index("id")
    .loc[ids_tendencia]
    .reset_index()
)

pronostico_tendencia_largo = pd.DataFrame({
    "item_id": np.repeat(
        filas_tendencia["item_id"].to_numpy(),
        28
    ),
    "store_id": np.repeat(
        filas_tendencia["store_id"].to_numpy(),
        28
    ),
    "macrogrupo": "Tendencia",
    "date": np.tile(
        calendario_futuro["date"].to_numpy(),
        len(filas_tendencia)
    ),
    "demanda_pronosticada": predicciones_tendencia.ravel(),
    "modelo": "XGBoost"
})

ruta_forecast_tendencia = ruta_outputs / "forecast_tendencia.csv"

pronostico_tendencia_largo.to_csv(
    ruta_forecast_tendencia,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo:", ruta_forecast_tendencia)
print("Filas:", len(pronostico_tendencia_largo))
print(pronostico_tendencia_largo.head())

Archivo: C:\Users\yeral\Documents\M5 Walmart\outputs\forecast_tendencia.csv
Filas: 62972
         item_id store_id macrogrupo       date  demanda_pronosticada   modelo
0  HOBBIES_1_001     CA_1  Tendencia 2016-04-25              0.964702  XGBoost
1  HOBBIES_1_001     CA_1  Tendencia 2016-04-26              0.900524  XGBoost
2  HOBBIES_1_001     CA_1  Tendencia 2016-04-27              0.900524  XGBoost
3  HOBBIES_1_001     CA_1  Tendencia 2016-04-28              0.920825  XGBoost
4  HOBBIES_1_001     CA_1  Tendencia 2016-04-29              1.001001  XGBoost


In [45]:
# Convertir el pronóstico Sin patrón dominante a formato largo y guardarlo como CSV
# para utilizarlo posteriormente en GitHub y en la aplicación Streamlit.

filas_sin_patron = sales.loc[
    sales["id"].isin(ids_sin_patron),
    ["id", "item_id", "store_id"]
].copy()

# Mantener el mismo orden utilizado para generar las predicciones.
filas_sin_patron = (
    filas_sin_patron
    .set_index("id")
    .loc[ids_sin_patron]
    .reset_index()
)

pronostico_sin_patron_largo = pd.DataFrame({
    "item_id": np.repeat(
        filas_sin_patron["item_id"].to_numpy(),
        28
    ),
    "store_id": np.repeat(
        filas_sin_patron["store_id"].to_numpy(),
        28
    ),
    "macrogrupo": "Sin patrón dominante",
    "date": np.tile(
        calendario_futuro["date"].to_numpy(),
        len(filas_sin_patron)
    ),
    "demanda_pronosticada": predicciones_sin_patron.ravel(),
    "modelo": "XGBoost"
})

ruta_forecast_sin_patron = ruta_outputs / "forecast_sin_patron.csv"

pronostico_sin_patron_largo.to_csv(
    ruta_forecast_sin_patron,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo:", ruta_forecast_sin_patron)
print("Filas:", len(pronostico_sin_patron_largo))
print(pronostico_sin_patron_largo.head())

Archivo: C:\Users\yeral\Documents\M5 Walmart\outputs\forecast_sin_patron.csv
Filas: 779184
         item_id store_id            macrogrupo       date  \
0  HOBBIES_1_002     CA_1  Sin patrón dominante 2016-04-25   
1  HOBBIES_1_002     CA_1  Sin patrón dominante 2016-04-26   
2  HOBBIES_1_002     CA_1  Sin patrón dominante 2016-04-27   
3  HOBBIES_1_002     CA_1  Sin patrón dominante 2016-04-28   
4  HOBBIES_1_002     CA_1  Sin patrón dominante 2016-04-29   

   demanda_pronosticada   modelo  
0              0.180103  XGBoost  
1              0.171216  XGBoost  
2              0.158133  XGBoost  
3              0.091040  XGBoost  
4              0.099730  XGBoost  


In [46]:
# Verificar que los tres archivos de pronóstico tengan la estructura y cantidad
# de registros esperadas antes de utilizarlos en GitHub y Streamlit.

archivos_forecast = [
    ruta_outputs / "forecast_estacional.csv",
    ruta_outputs / "forecast_tendencia.csv",
    ruta_outputs / "forecast_sin_patron.csv"
]

for archivo in archivos_forecast:
    df = pd.read_csv(archivo)

    print("\n", archivo.name)
    print("Filas:", len(df))
    print("Columnas:", df.columns.tolist())
    print("Fecha inicial:", df["date"].min())
    print("Fecha final:", df["date"].max())
    print("Modelos:", df["modelo"].unique())
    print("Valores nulos:", df.isna().sum().sum())


 forecast_estacional.csv
Filas: 11564
Columnas: ['item_id', 'store_id', 'macrogrupo', 'date', 'demanda_pronosticada', 'modelo']
Fecha inicial: 2016-04-25
Fecha final: 2016-05-22
Modelos: <StringArray>
['XGBoost']
Length: 1, dtype: str
Valores nulos: 0

 forecast_tendencia.csv
Filas: 62972
Columnas: ['item_id', 'store_id', 'macrogrupo', 'date', 'demanda_pronosticada', 'modelo']
Fecha inicial: 2016-04-25
Fecha final: 2016-05-22
Modelos: <StringArray>
['XGBoost']
Length: 1, dtype: str
Valores nulos: 0

 forecast_sin_patron.csv
Filas: 779184
Columnas: ['item_id', 'store_id', 'macrogrupo', 'date', 'demanda_pronosticada', 'modelo']
Fecha inicial: 2016-04-25
Fecha final: 2016-05-22
Modelos: <StringArray>
['XGBoost']
Length: 1, dtype: str
Valores nulos: 0


In [47]:
# Revisar el tamaño de los archivos de pronóstico antes de incorporarlos
# al repositorio y utilizarlos como fuente de datos para la aplicación.

for archivo in archivos_forecast:
    tamano_mb = archivo.stat().st_size / (1024 ** 2)
    print(f"{archivo.name}: {tamano_mb:.2f} MB")

forecast_estacional.csv: 0.65 MB
forecast_tendencia.csv: 3.49 MB
forecast_sin_patron.csv: 52.91 MB
